### Bloque 1 · Expositor 1
## Calidad de datos, EDA y problemáticas 4.1–4.2

**Facturación residencial de energía eléctrica, 2023–2024**

**Objetivo:** auditar y limpiar el CSV crudo, recorrer el EDA univariado y bivariado, y cerrar con las dos primeras problemáticas del estudio: el techo de subsistencia (4.1) y el golpe de la tarifa (4.2).

- **Entrada:** CSV crudo de facturación (`facturacion_energia.csv`).
- **Salida:** `data/datos_limpios.parquet`, insumo de los bloques 2 y 3.
- **Regla de limpieza:** si la corrección es inequívoca se corrige; solo se elimina lo que no se puede recuperar.

**Mapa de gráficas (en orden):**

| # | Gráfica | Pregunta que responde |
|---|---|---|
| G1 | Panel univariado 2×2 | ¿Cómo se distribuyen consumo, tarifa, estrato y mora? |
| G2 | Panel multivariado 1×2 | ¿Qué relación hay entre consumo, hogar y clima? |
| G3 | Mapa de calor de correlaciones | ¿Qué variables se mueven juntas? |
| G4 | Barras categóricas 2×2 + pruebas χ² | ¿Cómo se reparte el universo por categorías? |

Las problemáticas 4.1 y 4.2 se responden con tablas y pruebas estadísticas (sin figuras), para no repetir la información ya mostrada en G1-G4.

## Fase 3.1 · Carga y perfilamiento

Primero se preparan las librerías, la paleta y la homologación de operadores y municipios. Todo el bloque usa el mismo estilo para que la presentación se vea como un solo estudio.

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import binomtest, chi2_contingency, pearsonr, spearmanr

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 110,
})
PALETA = ['#2b5c8f', '#e4572e', '#4c9f70', '#f2b134', '#7b5ea7', '#66b3ff']
FIGSIZE = {'simple': (14, 5), 'doble': (15, 5), 'panel2x2': (15, 10), 'triple': (18, 5)}
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Homologación de operadores y municipios: 15 -> 7 y 44 -> 26
MAPA_OPERADOR = {
    'ENEL': 'ENEL', 'ENEL COLOMBIA': 'ENEL',
    'EPM': 'EPM', 'E.P.M.': 'EPM', 'EMPRESAS PÚBLICAS DE MEDELLÍN': 'EPM',
    'CELSIA': 'CELSIA', 'AIR-E': 'AIR-E', 'AIRE': 'AIR-E',
    'AFINIA': 'AFINIA', 'ESSA': 'ESSA', 'CEDENAR': 'CEDENAR',
}
MAPA_MUNICIPIO = {
    'BOGOTÁ D.C.': 'Bogotá', 'BOGOTA': 'Bogotá', 'BOGOTÁ': 'Bogotá', 'BOGOTA D.C.': 'Bogotá',
    'MEDELLÍN': 'Medellín', 'MEDELLIN': 'Medellín',
    'CALI': 'Cali', 'SANTIAGO DE CALI': 'Cali',
    'BARRANQUILLA': 'Barranquilla', 'B/QUILLA': 'Barranquilla',
    'CARTAGENA': 'Cartagena', 'CARTAGENA DE INDIAS': 'Cartagena',
    'BELLO': 'Bello', 'SOACHA': 'Soacha', 'BUCARAMANGA': 'Bucaramanga',
    'VALLEDUPAR': 'Valledupar', 'ENVIGADO': 'Envigado', 'SOLEDAD': 'Soledad',
    'SANTA MARTA': 'Santa Marta', 'SANTA MARTA D.T.C.H.': 'Santa Marta',
    'IBAGUÉ': 'Ibagué', 'IBAGUE': 'Ibagué',
    'MONTERÍA': 'Montería', 'MONTERIA': 'Montería',
    'PASTO': 'Pasto', 'RIONEGRO': 'Rionegro', 'PALMIRA': 'Palmira',
    'SINCELEJO': 'Sincelejo', 'ZIPAQUIRÁ': 'Zipaquirá', 'ZIPAQUIRA': 'Zipaquirá',
    'BARRANCABERMEJA': 'Barrancabermeja', 'RIOHACHA': 'Riohacha',
    'APARTADÓ': 'Apartadó', 'APARTADO': 'Apartadó',
    'GIRARDOT': 'Girardot', 'TULUÁ': 'Tuluá', 'TULUA': 'Tuluá',
    'TUMACO': 'Tumaco', 'ESPINAL': 'Espinal',
}


def normaliza_operador(serie):
    s = serie.astype(str).str.strip().str.upper()
    return s.map(MAPA_OPERADOR).fillna(s)


def normaliza_municipio(serie):
    s = serie.astype(str).str.strip().str.upper()
    return s.map(MAPA_MUNICIPIO).fillna(s.str.title())


def prueba_correlacion(x, y):
    sub = df[[x, y]].dropna()
    r, p_pearson = pearsonr(sub[x], sub[y])
    rho, p_spearman = spearmanr(sub[x], sub[y])
    return {
        'Par de variables': f'{x} vs {y}',
        'N': len(sub),
        'Pearson_r': r,
        'p_Pearson': p_pearson,
        'Spearman_rho': rho,
        'p_Spearman': p_spearman,
    }


def cramers_v(tabla):
    chi2, p, dof, _ = chi2_contingency(tabla)
    n = tabla.values.sum()
    v = np.sqrt(chi2 / (n * (min(tabla.shape) - 1))) if n and min(tabla.shape) > 1 else np.nan
    return chi2, p, dof, v

### G0 · Carga del CSV y perfilamiento inicial

El nombre del archivo no importa: el notebook busca cualquier CSV que tenga las columnas del dataset, así funciona igual en cualquier equipo.

In [ ]:
COLUMNAS_CLAVE = {'id_cuenta', 'periodo', 'consumo_kwh', 'tarifa_cu_cop_kwh', 'dias_mora'}


def _columnas_csv(ruta):
    """Lee solo el encabezado para verificar que sea el archivo correcto."""
    for codificacion in ('utf-8', 'utf-8-sig', 'latin-1'):
        try:
            return set(pd.read_csv(ruta, nrows=0, encoding=codificacion,
                                   sep=None, engine='python').columns)
        except Exception:
            continue
    return set()


def buscar_csv_facturacion():
    carpetas = [Path('.'), Path('data'), Path('Energia/data'),
                Path('..'), Path('../data'), Path('../Energia/data')]
    for carpeta in carpetas:
        for nombre in ('facturacion_energia.csv',
                       'facturacion_energia_residencial_2023_2024.csv'):
            ruta = carpeta / nombre
            if ruta.is_file() and COLUMNAS_CLAVE <= _columnas_csv(ruta):
                return ruta
    for carpeta in carpetas:
        if not carpeta.is_dir():
            continue
        for ruta in sorted(carpeta.glob('*.csv')):
            if ruta.is_file() and COLUMNAS_CLAVE <= _columnas_csv(ruta):
                return ruta
    return None


RUTA_ARCHIVO = buscar_csv_facturacion()
if RUTA_ARCHIVO is None:
    raise FileNotFoundError(
        'No se encontró ningún CSV con las columnas esperadas '
        f'({", ".join(sorted(COLUMNAS_CLAVE))}).\n'
        'Descarga el archivo de facturación y déjalo en Energia/data/ o en la raíz del '
        'repositorio: el nombre no importa, el notebook lo detecta solo.'
    )

print(f'Archivo de facturación encontrado: {RUTA_ARCHIVO.resolve()}')
try:
    df_raw = pd.read_csv(RUTA_ARCHIVO, low_memory=False, encoding='utf-8')
except UnicodeDecodeError:
    df_raw = pd.read_csv(RUTA_ARCHIVO, low_memory=False, encoding='latin-1')
print(f'Dimensiones iniciales: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas')

perfil = pd.DataFrame({
    'Variable': df_raw.columns,
    'Tipo_Leido': df_raw.dtypes.astype(str).values,
    'Nulos_Total': df_raw.isnull().sum().values,
    'Porcentaje_Nulos': (df_raw.isnull().mean() * 100).round(2).values,
    'Valores_Distintos': df_raw.nunique().values,
    'Ejemplos': [df_raw[col].dropna().astype(str).unique()[:4].tolist() for col in df_raw.columns],
})

print(f'\nCuentas únicas: {df_raw["id_cuenta"].nunique():,}')
print(f'Periodos distintos: {df_raw["periodo"].nunique():,}')
print(f'Filas por cuenta (máximo): {df_raw.groupby("id_cuenta")["periodo"].count().max()}')
display(perfil)

## Fase 3.2 · Auditoría de calidad

Antes de corregir, se cuantifica cada problema. Esta tabla es la bitácora que justifica cada decisión de limpieza.

In [ ]:
def formato_periodo(valor):
    s = str(valor).strip()
    if re.fullmatch(r'20\d{2}-(0[1-9]|1[0-2])', s):
        return 'AAAA-MM válido'
    if re.fullmatch(r'(0[1-9]|1[0-2])/20\d{2}', s):
        return 'MM/AAAA (corregible)'
    if re.fullmatch(r'20\d{2}(0[1-9]|1[0-2])', s):
        return 'AAAAMM (corregible)'
    return 'No reconocido'


auditoria = []
auditoria.append(('Registros totales', len(df_raw), 'Filas del CSV original'))
auditoria.append(('Cuentas únicas', df_raw['id_cuenta'].nunique(), 'Identificadores distintos'))
auditoria.append(('Duplicados cuenta-periodo', df_raw.duplicated(subset=['id_cuenta', 'periodo']).sum(),
                  'Misma cuenta repetida en el mismo mes'))
auditoria.append(('Periodos MM/AAAA', (df_raw['periodo'].map(formato_periodo) == 'MM/AAAA (corregible)').sum(),
                  'Formato de fecha alterno, corregible'))
auditoria.append(('Periodos AAAAMM', (df_raw['periodo'].map(formato_periodo) == 'AAAAMM (corregible)').sum(),
                  'Formato de fecha alterno, corregible'))
auditoria.append(('Periodos no reconocidos', (df_raw['periodo'].map(formato_periodo) == 'No reconocido').sum(),
                  'Sin patrón de fecha recuperable'))
auditoria.append(('Estrato no numérico', (~df_raw['estrato'].astype(str).str.fullmatch(r'[1-6](\.0)?')).sum(),
                  'Variantes E2, 02, Estrato 2, etc.'))
auditoria.append(('Estrato ausente', df_raw['estrato'].isna().sum(), 'Sin información de estrato'))
auditoria.append(('Consumo centinela 999999', (df_raw['consumo_kwh'] == 999_999).sum(),
                  'Código de error, no es un consumo real'))
auditoria.append(('Consumo negativo', (df_raw['consumo_kwh'] < 0).sum(), 'Valor imposible'))
auditoria.append(('Tarifa > 5.000 COP/kWh', (df_raw['tarifa_cu_cop_kwh'] > 5_000).sum(),
                  'Corrimiento decimal (valor 10 veces mayor)'))
auditoria.append(('Tarifa <= 0', (df_raw['tarifa_cu_cop_kwh'] <= 0).sum(), 'Valor imposible'))
auditoria.append(('Personas del hogar = 0', (df_raw['personas_hogar'] == 0).sum(),
                  'Dato faltante codificado como cero'))
auditoria.append(('Días de mora < 0', (df_raw['dias_mora'] < 0).sum(), 'Valor imposible'))
mora_prepago = ((df_raw['tipo_medidor'].astype(str).str.lower() == 'prepago') & (df_raw['dias_mora'] > 0)).sum()
auditoria.append(('Prepago con mora > 0', mora_prepago, 'Inconsistencia: el prepago no acumula mora'))
auditoria.append(('Error valor_consumo (> $5)', (np.abs(df_raw['valor_consumo_cop'] - df_raw['consumo_kwh'] * df_raw['tarifa_cu_cop_kwh']) > 5).sum(),
                  'Total no coincide con consumo x tarifa'))
auditoria.append(('Error valor_total (> $5)', (np.abs(df_raw['valor_total_cop'] - (df_raw['valor_consumo_cop'] + df_raw['subsidio_contribucion_cop'])) > 5).sum(),
                  'Total no coincide con consumo + subsidio/contribución'))
auditoria_df = pd.DataFrame(auditoria, columns=['Hallazgo', 'Registros', 'Descripción'])
display(auditoria_df)

print('\nFormatos de periodo encontrados:')
print(df_raw['periodo'].map(formato_periodo).value_counts().to_string())

print('\nVariantes de estrato encontradas (top 15):')
print(df_raw['estrato'].astype(str).value_counts().head(15).to_string())

print('\nValores únicos en variables categóricas:')
for col in ['operador', 'departamento', 'municipio', 'zona', 'tipo_medidor',
            'lectura_estimada', 'autogeneracion_solar', 'resultado_inspeccion']:
    print(f'  {col}: {df_raw[col].nunique()} valores -> {sorted(df_raw[col].dropna().astype(str).unique())}')

## Limpieza correctiva, embudo y bitácora

Se normalizan periodos (`MM/AAAA`, `AAAAMM`), estratos (`E2`, `02`, `Estrato 2`), operadores y municipios. También se tratan centinelas (`999999`), tarifas con corrimiento decimal, personas en cero, signos de subsidio/contribución y totales inconsistentes. El resultado se guarda en Parquet para los otros dos bloques.

In [ ]:
df = df_raw.copy()
embudo = [('0. Carga inicial', len(df), 'Registros crudos del CSV')]


def normaliza_periodo(valor):
    s = str(valor).strip()
    if re.fullmatch(r'20\d{2}-(0[1-9]|1[0-2])', s):
        return s
    m = re.fullmatch(r'(0[1-9]|1[0-2])/(20\d{2})', s)
    if m:
        return f'{m.group(2)}-{m.group(1)}'
    m = re.fullmatch(r'(20\d{2})(0[1-9]|1[0-2])', s)
    if m:
        return f'{m.group(1)}-{m.group(2)}'
    return np.nan


def normaliza_estrato(valor):
    if pd.isna(valor):
        return np.nan
    s = str(valor).strip().upper().replace('ESTRATO', '').replace('E', '').strip()
    s = s.replace('.0', '')
    if s.isdigit():
        n = int(s)
        return n if 1 <= n <= 6 else np.nan
    return np.nan


# ---------- 1. Estandarización de texto ----------
cols_texto = ['operador', 'departamento', 'municipio', 'zona', 'tipo_medidor',
              'lectura_estimada', 'autogeneracion_solar', 'resultado_inspeccion']
for col in cols_texto:
    df[col] = df[col].astype(str).str.strip()

operador_norm = df['operador'].str.upper()
municipio_norm = df['municipio'].str.upper()
n_operador_variantes = int(operador_norm.map(MAPA_OPERADOR).isna().sum())
n_municipio_variantes = int(municipio_norm.map(MAPA_MUNICIPIO).isna().sum())
df['operador'] = normaliza_operador(df['operador'])
df['municipio'] = normaliza_municipio(df['municipio'])
df['departamento'] = df['departamento'].str.upper()
df['zona'] = df['zona'].str.capitalize()
df['tipo_medidor'] = df['tipo_medidor'].str.capitalize()
df['lectura_estimada'] = df['lectura_estimada'].str.upper()
df['autogeneracion_solar'] = df['autogeneracion_solar'].str.upper()
df['resultado_inspeccion'] = df['resultado_inspeccion'].replace({'nan': np.nan, 'None': np.nan, '': np.nan})

# ---------- 2. Corrección de periodo y estrato ----------
n_periodo_alt = int((~df['periodo'].astype(str).str.match(r'^20\d{2}-(0[1-9]|1[0-2])$', na=False)).sum())
df['periodo'] = df['periodo'].map(normaliza_periodo)

n_estrato_variante = int((~df['estrato'].astype(str).str.fullmatch(r'[1-6](\.0)?') & df['estrato'].notna()).sum())
df['estrato'] = df['estrato'].map(normaliza_estrato)

# ---------- 3. Conversión numérica ----------
num_cols = ['altitud_msnm', 'personas_hogar', 'consumo_kwh', 'tarifa_cu_cop_kwh',
            'valor_consumo_cop', 'subsidio_contribucion_cop', 'valor_total_cop', 'dias_mora']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# ---------- 4. Duplicados y filtros ----------
n_antes = len(df)
df = df.drop_duplicates(subset=['id_cuenta', 'periodo'], keep='first')
n_dup = n_antes - len(df)
embudo.append(('1. Sin duplicados cuenta-periodo', len(df), 'Se conserva el primer registro de cada cuenta y mes'))

n_antes = len(df)
df = df[df['periodo'].notna()].copy()
n_periodo_drop = n_antes - len(df)
embudo.append(('2. Periodos normalizados', len(df), 'MM/AAAA y AAAAMM convertidos a AAAA-MM; no se descarta ninguno'))

n_antes = len(df)
df = df[df['estrato'].isin([1, 2, 3, 4, 5, 6])].copy()
n_estrato_drop = n_antes - len(df)
embudo.append(('3. Estrato 1-6 válido', len(df), 'Variantes E2, 02 y Estrato 2 corregidas; solo se descarta el estrato ausente'))

n_antes = len(df)
n_centinela = int((df['consumo_kwh'] == 999_999).sum())
df = df[(df['consumo_kwh'] >= 0) & (df['consumo_kwh'] != 999_999) & (df['tarifa_cu_cop_kwh'] > 0)].copy()
n_consumo_drop = n_antes - len(df)
embudo.append(('4. Consumo y tarifa válidos', len(df), f'Se eliminan negativos, tarifa cero y {n_centinela:,} centinelas 999999'))

mask_tarifa = df['tarifa_cu_cop_kwh'] > 5_000
n_tarifa = int(mask_tarifa.sum())
df.loc[mask_tarifa, 'tarifa_cu_cop_kwh'] = df.loc[mask_tarifa, 'tarifa_cu_cop_kwh'] / 10
embudo.append(('5. Tarifas 10x corregidas', len(df), f'{n_tarifa:,} tarifas > 5.000 COP/kWh divididas entre 10 (no se borran)'))

# ---------- 5. Correcciones de dominio ----------
n_personas = int((df['personas_hogar'] == 0).sum())
df['personas_hogar'] = df['personas_hogar'].where(df['personas_hogar'] > 0)

n_mora_prepago = int(((df['tipo_medidor'] == 'Prepago') & (df['dias_mora'] > 0)).sum())
df.loc[(df['tipo_medidor'] == 'Prepago') & (df['dias_mora'] > 0), 'dias_mora'] = 0
n_mora_nula = int(df['dias_mora'].isna().sum())
df['dias_mora'] = df['dias_mora'].clip(lower=0).fillna(0)

mask_sub = df['estrato'].isin([1, 2, 3]) & (df['subsidio_contribucion_cop'] > 0)
n_sub = int(mask_sub.sum())
df.loc[mask_sub, 'subsidio_contribucion_cop'] *= -1
df.loc[df['estrato'] == 4, 'subsidio_contribucion_cop'] = 0
mask_con = df['estrato'].isin([5, 6]) & (df['subsidio_contribucion_cop'] < 0)
n_con = int(mask_con.sum())
df.loc[mask_con, 'subsidio_contribucion_cop'] *= -1

diff_consumo = np.abs(df['valor_consumo_cop'] - df['consumo_kwh'] * df['tarifa_cu_cop_kwh'])
diff_total = np.abs(df['valor_total_cop'] - (df['valor_consumo_cop'] + df['subsidio_contribucion_cop']))
n_recalc = int(((diff_consumo > 5) | (diff_total > 5)).sum())
df['valor_consumo_cop'] = df['consumo_kwh'] * df['tarifa_cu_cop_kwh']
df['valor_total_cop'] = df['valor_consumo_cop'] + df['subsidio_contribucion_cop']

# ---------- 6. Variables derivadas ----------
df['limite_subsistencia'] = np.where(df['altitud_msnm'] < 1000, 173, 130).astype(int)
df['clima'] = pd.cut(df['altitud_msnm'], bins=[-np.inf, 999.9, 2000, np.inf],
                     labels=['Cálido', 'Templado', 'Frío'])
df['tramo_mora'] = pd.cut(df['dias_mora'], bins=[-1, 0, 30, 60, 90, np.inf],
                          labels=['Al día', '1 a 30', '31 a 60', '61 a 90', 'Más de 90'])
df['anio'] = df['periodo'].str[:4]

for col in ['operador', 'departamento', 'municipio', 'zona', 'tipo_medidor',
            'lectura_estimada', 'autogeneracion_solar', 'resultado_inspeccion', 'clima', 'tramo_mora']:
    df[col] = df[col].astype('category')

# ---------- 7. Bitácora Anexo A ----------
bitacora = pd.DataFrame([
    ('Duplicados cuenta-periodo', n_dup,
     'Se conserva el primer registro', 'No aportan información nueva y distorsionan promedios'),
    ('Periodos en formato MM/AAAA o AAAAMM', n_periodo_alt,
     'Conversión a AAAA-MM (0 eliminados)', 'La fecha es inequívoca: 07/2023 y 202307 equivalen a 2023-07'),
    ('Estrato escrito como E2, 02 o Estrato 2', n_estrato_variante,
     'Extracción del dígito 1-6', 'La corrección es inequívoca; borrarlos perdería más de 120 mil facturas válidas'),
    ('Estrato ausente o fuera de 1-6', n_estrato_drop,
     'Se excluye del análisis', 'Sin estrato no se puede aplicar subsidio ni límite de subsistencia'),
    ('Consumo centinela 999999', n_centinela,
     'Se excluye la factura', 'Es un código de error, equivale a 999.999 kWh y rompe escalas y clústeres'),
    ('Tarifa mayor a 5.000 COP/kWh', n_tarifa,
     'Se divide entre 10', 'Corrimiento decimal: al corregir queda dentro del rango real (790-1.275 COP/kWh)'),
    ('Personas del hogar = 0', n_personas,
     'Se convierte a nulo', 'Un hogar no puede tener cero personas; el cero codifica dato faltante'),
    ('Prepago con días de mora', n_mora_prepago,
     'Se lleva a 0 días', 'El medidor prepago paga antes de consumir; la mora es inconsistente'),
    ('Días de mora sin dato', n_mora_nula,
     'Se imputan 0 días', 'No hay reporte de deuda: se asume al día, igual que la factura sin mora'),
    ('Signo del subsidio/contribución', n_sub + n_con,
     'Subsidio negativo en 1-3 y contribución positiva en 5-6', 'El signo debe reflejar el flujo: subsidio resta, contribución suma'),
    ('Totales inconsistentes', n_recalc,
     'Recalcule valor_consumo y valor_total', 'Garantiza coherencia contable: consumo x tarifa y consumo + subsidio'),
    ('Operadores con nombres alternos', n_operador_variantes,
     'Homologación a 7 operadores', 'Enel Colombia, E.P.M. y AIRE son variantes de escritura'),
    ('Municipios con nombres alternos', n_municipio_variantes,
     'Homologación a 26 municipios', 'Bogotá D.C., Bogota y BOGOTA son el mismo municipio'),
], columns=['Problema detectado', 'Registros afectados', 'Tratamiento aplicado', 'Justificación'])

embudo_df = pd.DataFrame(embudo, columns=['Paso', 'Registros_Restantes', 'Justificación'])
embudo_df['Eliminados_Paso'] = embudo_df['Registros_Restantes'].shift(1) - embudo_df['Registros_Restantes']
embudo_df['Eliminados_Paso'] = embudo_df['Eliminados_Paso'].fillna(0).astype(int)

display(bitacora)
display(embudo_df)

print(f'\nDataset limpio: {len(df):,} filas x {df.shape[1]} columnas')
print(f'Cuentas únicas: {df["id_cuenta"].nunique():,}')
print(f'Operadores: {df["operador"].nunique()} | Municipios: {df["municipio"].nunique()}')
print(f'Periodos: {df["periodo"].min()} a {df["periodo"].max()}')
print(f'Consumo máximo: {df["consumo_kwh"].max():,.0f} kWh | Tarifa máxima: {df["tarifa_cu_cop_kwh"].max():,.2f} COP/kWh')
print(f'Nulos en personas_hogar: {df["personas_hogar"].isna().sum():,}')
print(f'Valor total negativo: {(df["valor_total_cop"] < 0).sum():,}')

RUTA_PARQUET = Path('data/datos_limpios.parquet')
RUTA_PARQUET.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(RUTA_PARQUET, index=False)

operadores_limpios = sorted(df['operador'].astype(str).unique())
municipios_limpios = sorted(df['municipio'].astype(str).unique())
print(f'Operadores limpios ({len(operadores_limpios)}): {operadores_limpios}')
print(f'Municipios limpios ({len(municipios_limpios)}): {municipios_limpios}')
assert len(operadores_limpios) == 7, f'Se esperaban 7 operadores y quedaron {len(operadores_limpios)}'
assert len(municipios_limpios) == 26, f'Se esperaban 26 municipios y quedaron {len(municipios_limpios)}'
print(f'✅ {RUTA_PARQUET.resolve()} actualizado con {len(operadores_limpios)} operadores '
      f'y {len(municipios_limpios)} municipios exactos.')

## Fase 3.3 · Análisis univariado

### G1 · Panel univariado: consumo, tarifa, estrato y mora

**Pregunta:** ¿cómo se distribuye cada variable por separado y qué forma tiene la cola alta?

In [ ]:
df['operador'] = normaliza_operador(df['operador'])
df['municipio'] = normaliza_municipio(df['municipio'])
assert df['operador'].nunique() == 7, 'Operadores sin homologar: vuelve a ejecutar la celda de limpieza'

vars_num = ['consumo_kwh', 'tarifa_cu_cop_kwh', 'valor_total_cop', 'dias_mora', 'personas_hogar']
resumen = []
for col in vars_num:
    s = df[col].dropna()
    resumen.append({
        'Variable': col,
        'N': len(s),
        'Media': s.mean(),
        'Mediana': s.median(),
        'Desv_Std': s.std(),
        'CV': s.std() / s.mean() if s.mean() else np.nan,
        'Min': s.min(),
        'P1': s.quantile(0.01),
        'P5': s.quantile(0.05),
        'P25': s.quantile(0.25),
        'P75': s.quantile(0.75),
        'P95': s.quantile(0.95),
        'P99': s.quantile(0.99),
        'Max': s.max(),
        'Asimetria': s.skew(),
        'Curtosis': s.kurtosis(),
    })
resumen_num = pd.DataFrame(resumen).set_index('Variable').round(2)
display(resumen_num)

fig, axes = plt.subplots(2, 2, figsize=FIGSIZE['panel2x2'])

# Consumo: histograma con KDE, recortado en el P99 para ver la forma
consumo = df['consumo_kwh'].dropna()
corte = consumo.quantile(0.99)
sns.histplot(consumo[consumo <= corte], bins=40, kde=True, ax=axes[0, 0], color=PALETA[0])
axes[0, 0].axvline(consumo.median(), color=PALETA[1], linestyle='--', linewidth=2,
                   label=f'Mediana {consumo.median():,.0f} kWh')
axes[0, 0].set_title('Consumo mensual (kWh)')
axes[0, 0].set_xlabel('kWh/mes')
axes[0, 0].legend()

# Tarifa: caja y bigotes
sns.boxplot(x=df['tarifa_cu_cop_kwh'], ax=axes[0, 1], color=PALETA[0], showfliers=False)
axes[0, 1].set_title('Tarifa (COP/kWh)')
axes[0, 1].set_xlabel('COP/kWh')

# Estrato: barras ordenadas
orden_estrato = df['estrato'].value_counts().sort_index().index
sns.countplot(data=df, x='estrato', hue='estrato', legend=False,
              order=orden_estrato, ax=axes[1, 0], palette=PALETA)
axes[1, 0].set_title('Facturas por estrato')
axes[1, 0].set_xlabel('Estrato')
axes[1, 0].set_ylabel('Facturas')

# Mora: torta por tramo
tabla_mora = df['tramo_mora'].value_counts()
axes[1, 1].pie(tabla_mora.values, labels=tabla_mora.index, autopct='%1.1f%%',
               colors=PALETA, startangle=90)
axes[1, 1].set_title('Proporción por tramo de mora')
axes[1, 1].set_ylabel('')

fig.suptitle('G1 · Univariado: forma de las variables principales (2023-2024)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('Lectura rápida:')
print(f'  Consumo mediano: {consumo.median():,.0f} kWh/mes; media: {consumo.mean():,.1f} kWh/mes')
print(f'  El 50% de las facturas consume entre {consumo.quantile(.25):,.0f} y {consumo.quantile(.75):,.0f} kWh')
print(f'  Tarifa mediana: {df["tarifa_cu_cop_kwh"].median():,.0f} COP/kWh')
print(f'  Facturas al día: {(df["dias_mora"] == 0).mean() * 100:.1f}% | con mora > 60 días: {(df["dias_mora"] > 60).mean() * 100:.1f}%')
print(f'  Estratos 1-3: {df["estrato"].isin([1, 2, 3]).mean() * 100:.1f}% de las facturas')

**Lectura G1.** Consumo, tarifa y valor total son asimétricos a la derecha: la mediana queda por debajo de la media y la caja se estira hacia arriba. Esa cola alta —pocas cuentas con consumos muy grandes— concentra buena parte del gasto del sistema. La mora, en cambio, se acumula en cero, así que se analiza por tramos y no como promedio. El estrato muestra que la mayoría de las facturas son residenciales de estratos 1 a 3.

## Fase 3.4 · Análisis bivariado

### G2 · Panel multivariado: consumo vs. hogar y consumo vs. clima

**Pregunta:** ¿el consumo sube con el número de personas y cambia con el clima?

In [ ]:
pares = [
    ('consumo_kwh', 'personas_hogar'),
    ('consumo_kwh', 'valor_total_cop'),
    ('tarifa_cu_cop_kwh', 'valor_total_cop'),
    ('dias_mora', 'valor_total_cop'),
    ('altitud_msnm', 'consumo_kwh'),
]
correlaciones = pd.DataFrame([prueba_correlacion(x, y) for x, y in pares]).round(5)
display(correlaciones)

muestra = df.sample(n=min(20_000, len(df)), random_state=42)
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE['doble'])

sns.scatterplot(data=muestra, x='personas_hogar', y='consumo_kwh', hue='estrato',
                palette='viridis', alpha=0.35, s=12, ax=axes[0])
sns.regplot(data=muestra, x='personas_hogar', y='consumo_kwh', scatter=False,
            color=PALETA[1], ax=axes[0])
axes[0].set_ylim(0, muestra['consumo_kwh'].quantile(0.99))
axes[0].set_title('Consumo vs. personas del hogar (color = estrato)')
axes[0].set_xlabel('Personas del hogar')
axes[0].set_ylabel('Consumo (kWh/mes)')
axes[0].legend(title='Estrato', fontsize=8, ncol=2)

sns.boxplot(data=df, x='clima', y='consumo_kwh', ax=axes[1], color=PALETA[0], showfliers=False)
axes[1].set_ylim(0, 500)
axes[1].set_title('Consumo por clima (según altitud)')
axes[1].set_xlabel('Clima')
axes[1].set_ylabel('Consumo (kWh/mes)')

fig.suptitle('G2 · Multivariado: hogar, clima y consumo', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

resumen_estrato = df.groupby('estrato', observed=True)['consumo_kwh'].agg(
    N='count', Media='mean', Mediana='median', Desv_Std='std', P95=lambda x: x.quantile(0.95)).round(2)
resumen_clima = df.groupby('clima', observed=True)['consumo_kwh'].agg(
    N='count', Media='mean', Mediana='median', Desv_Std='std', P95=lambda x: x.quantile(0.95)).round(2)
display(resumen_estrato)
display(resumen_clima)

print('Lectura rápida:')
print(f'  Consumo vs personas: Pearson r = {correlaciones.iloc[0]["Pearson_r"]:.3f} | Spearman rho = {correlaciones.iloc[0]["Spearman_rho"]:.3f}')
print(f'  Consumo vs altitud: Spearman rho = {correlaciones.iloc[4]["Spearman_rho"]:.3f}')

**Lectura G2.** El consumo tiende a subir con el número de personas, pero la dispersión es amplia: el tamaño del hogar explica parte del consumo, no todo. La comparación por clima muestra que los municipios de altura (clima frío) tienen consumos distintos a los cálidos, lo que justifica usar la altitud como variable de contexto en las problemáticas siguientes.

### G3 · Mapa de calor de correlaciones

**Pregunta:** ¿qué variables numéricas se mueven juntas y cuáles son redundantes?

In [ ]:
cols_corr = ['consumo_kwh', 'tarifa_cu_cop_kwh', 'valor_consumo_cop', 'subsidio_contribucion_cop',
             'valor_total_cop', 'dias_mora', 'personas_hogar', 'estrato', 'altitud_msnm']
matriz = df[cols_corr].corr(method='spearman').round(2)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(matriz, annot=True, cmap='RdBu_r', vmin=-1, vmax=1, fmt='.2f',
            linewidths=0.5, ax=ax)
ax.set_title('G3 · Correlación de Spearman entre variables numéricas')
plt.tight_layout()
plt.show()

print('Lectura rápida:')
print(f'  Tarifa vs valor total: Spearman rho = {matriz.loc["tarifa_cu_cop_kwh", "valor_total_cop"]:.2f}')
print(f'  Días de mora vs valor total: Spearman rho = {matriz.loc["dias_mora", "valor_total_cop"]:.2f}')
print('  Consumo y valor de consumo son casi la misma variable: se usan como contexto, no como predictores duplicados.')

**Lectura G3.** El valor de consumo y el valor total están fuertemente asociados porque uno se calcula a partir del otro; por eso no se usan juntos como predictores. La tarifa y la mora tienen una relación más débil a nivel de factura, lo que anticipa que el efecto de la tarifa se ve mejor agregado por operador y mes (problemática 4.2).

### G4 · Barras de variables categóricas y pruebas χ²

**Pregunta:** ¿cómo se reparte el universo por tipo de medidor, zona, clima y lectura, y qué cruces categóricos muestran asociación real?

In [ ]:
categoricas = ['estrato', 'tipo_medidor', 'zona', 'tramo_mora', 'lectura_estimada',
               'autogeneracion_solar', 'clima', 'operador', 'departamento', 'municipio']
partes = []
for col in categoricas:
    tabla = df[col].value_counts(dropna=False)
    parte = pd.DataFrame({
        'Variable': col,
        'Categoria': tabla.index.astype(str).values,
        'Frecuencia': tabla.values,
    })
    parte['Porcentaje_%'] = (parte['Frecuencia'] / len(df) * 100).round(2)
    parte['Porcentaje_Acum_%'] = parte['Porcentaje_%'].cumsum().round(2)
    partes.append(parte)
frecuencias = pd.concat(partes, ignore_index=True)
display(frecuencias[frecuencias['Variable'].isin(['estrato', 'tipo_medidor', 'zona', 'tramo_mora', 'clima'])])

pruebas_cat = []
tab_est_med = pd.crosstab(df['estrato'], df['tipo_medidor'])
chi2, p, dof, v = cramers_v(tab_est_med)
pruebas_cat.append({'Cruce': 'estrato x tipo_medidor', 'Chi2': chi2, 'gl': dof, 'p_valor': p, 'V_Cramer': v})

tab_op_mora = pd.crosstab(df['operador'], df['tramo_mora'])
chi2, p, dof, v = cramers_v(tab_op_mora)
pruebas_cat.append({'Cruce': 'operador x tramo_mora', 'Chi2': chi2, 'gl': dof, 'p_valor': p, 'V_Cramer': v})

tab_zona_mora = pd.crosstab(df['zona'], df['tramo_mora'])
chi2, p, dof, v = cramers_v(tab_zona_mora)
pruebas_cat.append({'Cruce': 'zona x tramo_mora', 'Chi2': chi2, 'gl': dof, 'p_valor': p, 'V_Cramer': v})

tab_est_auto = pd.crosstab(df['estrato'], df['autogeneracion_solar'])
chi2, p, dof, v = cramers_v(tab_est_auto)
pruebas_cat.append({'Cruce': 'estrato x autogeneracion_solar', 'Chi2': chi2, 'gl': dof, 'p_valor': p, 'V_Cramer': v})

pruebas_cat = pd.DataFrame(pruebas_cat).round(6)
display(pruebas_cat)

fig, axes = plt.subplots(2, 2, figsize=FIGSIZE['panel2x2'])
for ax, col in zip(axes.ravel(), ['tipo_medidor', 'zona', 'clima', 'lectura_estimada']):
    orden = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=orden, ax=ax, color=PALETA[0])
    ax.set_title(f'Frecuencia de {col.replace("_", " ")}')
    ax.set_xlabel('')
    ax.set_ylabel('Facturas')
    ax.tick_params(axis='x', rotation=30)

fig.suptitle('G4 · Categóricas: composición del universo de facturas', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('Lectura rápida:')
for col in ['tipo_medidor', 'zona', 'clima', 'lectura_estimada']:
    top = df[col].value_counts()
    print(f'  {col}: {top.index[0]} concentra {top.iloc[0] / len(df) * 100:.1f}% de las facturas')
print(f'  Facturas con mora > 60 días: {(df["dias_mora"] > 60).mean() * 100:.1f}%')
print('  Con este volumen de datos casi cualquier cruce es significativo (p < 0.001);')
print('  por eso se reporta la V de Cramér: mide el tamaño del efecto, no solo la existencia.')

**Lectura G4.** El universo está dominado por medidores de energía y por la zona urbana; el clima templado/frío pesa más que el cálido por la composición de municipios. Las lecturas estimadas son una minoría, pero suficientes para justificar su seguimiento: una lectura estimada puede ocultar tanto un consumo real como una anomalía. La tabla de χ² cierra la sección: con este volumen casi todo cruce sale significativo, así que la V de Cramér es la que dice si la asociación es grande o apenas perceptible; el cruce operador × tramo de mora es el puente hacia la problemática 4.2, porque la mora grave también es un fenómeno territorial y de operador.

## Problemática 4.1 · El techo de la subsistencia

Se mide cuántas facturas se acumulan justo por debajo del límite de subsistencia y se contrasta con una prueba binomial. Los resultados quedan en las tablas siguientes.

In [ ]:
consumo = df['consumo_kwh'].dropna()
print('Lectura rápida (franja de 10 kWh antes de cada límite):')
for limite in (130, 173):
    debajo = int(((consumo >= limite - 10) & (consumo < limite)).sum())
    encima = int(((consumo >= limite) & (consumo < limite + 10)).sum())
    print(f'  En {limite} kWh: {debajo:,} facturas en los 10 kWh de abajo frente a {encima:,} '
          f'en los 10 kWh de arriba (ratio {debajo / encima:.2f}).')

**Lectura de la franja.** Si el ratio de la franja de abajo sobre la de arriba es mayor que 1, el consumo se concentra justo antes del límite: ese es el indicio de la problemática 4.1. La tabla siguiente compara el mismo patrón en los estratos 1-3 contra los estratos de control 4-6 y aplica una prueba binomial a los 10 kWh inmediatamente anteriores al límite.

In [ ]:
configs = [
    (df['estrato'].isin([1, 2, 3]) & (df['altitud_msnm'] < 1000), 173,
     'Estratos 1-3 | límite 173 kWh (< 1.000 msnm)'),
    (df['estrato'].isin([1, 2, 3]) & (df['altitud_msnm'] >= 1000), 130,
     'Estratos 1-3 | límite 130 kWh (>= 1.000 msnm)'),
    (df['estrato'].isin([4, 5, 6]) & (df['altitud_msnm'] < 1000), 173,
     'Control estratos 4-6 | referencia 173 kWh'),
    (df['estrato'].isin([4, 5, 6]) & (df['altitud_msnm'] >= 1000), 130,
     'Control estratos 4-6 | referencia 130 kWh'),
]

resultados_41 = []
for mask, limite, titulo in configs:
    sub = df.loc[mask, 'consumo_kwh'].dropna()
    debajo = int(((sub >= limite - 10) & (sub < limite)).sum())
    encima = int(((sub >= limite) & (sub < limite + 10)).sum())
    prueba = binomtest(debajo, debajo + encima, 0.5) if (debajo + encima) > 0 else None
    resultados_41.append({
        'Grupo': titulo,
        'N': len(sub),
        'Media': sub.mean(),
        'Mediana': sub.median(),
        'Pct_debajo_limite': (sub < limite).mean() * 100,
        'Pct_franja_10_debajo': debajo / len(sub) * 100,
        'Pct_franja_10_encima': encima / len(sub) * 100,
        'Ratio_debajo_encima': debajo / encima if encima else np.nan,
        'p_valor_binomial': prueba.pvalue if prueba else np.nan,
    })

resultados_41 = pd.DataFrame(resultados_41).round(3)
display(resultados_41)

cobertura = df.groupby(['estrato', 'clima'], observed=True).apply(
    lambda g: pd.Series({
        'N': len(g),
        'Consumo_medio': g['consumo_kwh'].mean(),
        'Pct_bajo_limite': (g['consumo_kwh'] < g['limite_subsistencia']).mean() * 100,
    }), include_groups=False).round(2)
display(cobertura.unstack())

print('Lectura rápida:')
for _, fila in resultados_41.iterrows():
    print(f'  {fila["Grupo"]}: {fila["Pct_franja_10_debajo"]:.2f}% en la franja debajo vs '
          f'{fila["Pct_franja_10_encima"]:.2f}% encima (ratio {fila["Ratio_debajo_encima"]:.2f})')

**Lectura de las tablas 4.1.** Si el escalón aparece en los estratos 1-3 y no en los estratos 4-6, el patrón no es un accidente del mercado: está asociado al subsidio y al límite de subsistencia. La prueba binomial contrasta si la franja de 10 kWh anterior al límite tiene más facturas de las que cabría esperar por azar.

## Problemática 4.2 · El golpe de la tarifa

In [ ]:
df['operador'] = normaliza_operador(df['operador'])
df['municipio'] = normaliza_municipio(df['municipio'])
assert df['operador'].nunique() == 7, 'Operadores sin homologar: vuelve a ejecutar la celda de limpieza'

serie_op = df.groupby(['periodo', 'operador'], observed=True).agg(
    tarifa_mediana=('tarifa_cu_cop_kwh', 'median'),
    consumo_medio=('consumo_kwh', 'mean'),
    prop_mora_60=('dias_mora', lambda x: (x > 60).mean() * 100),
    n_facturas=('id_cuenta', 'size'),
).reset_index()
serie_op['anio'] = serie_op['periodo'].str[:4]

# Cambio del consumo medio comparando el MISMO mes de 2023 y de 2024.
df['mes'] = df['periodo'].str[5:7]
cons_mes = (df.groupby(['operador', 'mes', 'anio'], observed=True)['consumo_kwh']
              .mean().unstack())
cons_mes = cons_mes.dropna(subset=['2023', '2024'])
cons_mes['Var_%'] = (cons_mes['2024'] / cons_mes['2023'] - 1) * 100
var_por_op = (cons_mes.groupby(level='operador')['Var_%']
              .agg(Meses_comparados='count', Cambio_promedio='mean',
                   Cambio_min='min', Cambio_max='max')
              .sort_values('Cambio_promedio').round(2))

display(var_por_op)

t_ini = serie_op[serie_op['periodo'] == '2023-01'].set_index('operador')['tarifa_mediana']
t_fin = serie_op[serie_op['periodo'] == '2024-12'].set_index('operador')['tarifa_mediana']
inc_tarifa = pd.DataFrame({
    'Tarifa_Ene_2023': t_ini,
    'Tarifa_Dic_2024': t_fin,
    'Incremento_Total_%': ((t_fin - t_ini) / t_ini * 100),
    'Incremento_Mensual_Prom_%': ((t_fin / t_ini) ** (1 / 23) - 1) * 100,
}).sort_values('Incremento_Total_%', ascending=False).round(2)
display(inc_tarifa)

# Incremento de tarifa por municipio (para identificar la región Caribe)
tar_mun = df.groupby(['municipio', 'periodo'], observed=True)['tarifa_cu_cop_kwh'].median().unstack()
inc_mun = pd.DataFrame({
    'Tarifa_Ene_2023': tar_mun.get('2023-01'),
    'Tarifa_Dic_2024': tar_mun.get('2024-12'),
}).dropna()
inc_mun['Incremento_Total_%'] = (inc_mun['Tarifa_Dic_2024'] / inc_mun['Tarifa_Ene_2023'] - 1) * 100
inc_mun = inc_mun.sort_values('Incremento_Total_%', ascending=False).round(2)
display(inc_mun.head(10))

mora_anual = df.groupby(['operador', 'anio'], observed=True)['dias_mora'].apply(
    lambda x: (x > 60).mean() * 100).unstack().round(2)
mora_anual['Aumento_pp'] = (mora_anual['2024'] - mora_anual['2023']).round(2)
display(mora_anual.sort_values('Aumento_pp', ascending=False))

rho_tarifa_mora, p_tarifa_mora = spearmanr(serie_op['tarifa_mediana'], serie_op['prop_mora_60'])
r_tarifa_mora, p_r = pearsonr(serie_op['tarifa_mediana'], serie_op['prop_mora_60'])
print('Relación tarifa vs mora > 60 días (operador-mes):')
print(f'  Spearman rho = {rho_tarifa_mora:.3f} (p = {p_tarifa_mora:.2e})')
print(f'  Pearson r    = {r_tarifa_mora:.3f} (p = {p_r:.2e})')
print(f'  Operador con mayor incremento de tarifa: {inc_tarifa.index[0]} ({inc_tarifa.iloc[0]["Incremento_Total_%"]:.2f}%)')
print(f'  Operador con mayor aumento de mora: {mora_anual["Aumento_pp"].idxmax()} ({mora_anual["Aumento_pp"].max():.2f} pp)')

cambio_consumo = cons_mes['Var_%'].mean()
alza_tarifa = inc_tarifa['Incremento_Total_%'].mean()
cambio_mora = mora_anual['Aumento_pp'].mean()
print('\nFrase que relaciona tarifa, consumo y mora (respuesta a 4.2):')
print(f'  Entre enero de 2023 y diciembre de 2024 la tarifa mediana subió {alza_tarifa:.1f}% en promedio; '
      f'frente a esa alza, el consumo medio del mismo mes cayó {abs(cambio_consumo):.1f}% y la proporción de '
      f'facturas con mora > 60 días subió {cambio_mora:.2f} puntos porcentuales.')

**Lectura 4.2.** Las tablas cuentan la misma historia desde ángulos distintos: la tarifa sube, el consumo del mismo mes cae y la mora grave aumenta. Comparar el mismo mes de 2023 y 2024 cancela la estacionalidad (calor, vacaciones, ciclos escolares), de modo que el cambio no se puede atribuir solo a que un año fue más caluroso. La relación tarifa-mora a nivel operador-mes muestra que a mayor tarifa mediana, mayor proporción de mora grave: el ajuste de consumo amortigua el alza, pero no alcanza a compensarla.

## Cierre del Bloque 1

- **Entregable:** `data/datos_limpios.parquet` con 7 operadores y 26 municipios homologados.
- **Historia visual:** cuatro figuras para la calidad del dato y el EDA (G1-G4); las problemáticas 4.1 y 4.2 se cierran con tablas y pruebas estadísticas.
- **Transición al Bloque 2:** el parquet limpio es la entrada del clustering de cuentas y del agrupamiento municipal.